# Lezione 8 — Analisi del modello

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife, basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione: 90 minuti**

## Obiettivi di apprendimento di questa lezione

- Inspezionare i risultati del modello oltre alla semplice accuratezza (matrice di confusione, precision, recall, F1-score)
- Identificare su quali classi il modello sbaglia più frequentemente
- Visualizzare gli errori per imparare dove il modello ha problemi
- Usare tecniche di spiegabilità per capire quali feature il modello sta usando

Questo notebook accompagna il capitolo 08 del manuale del corso. Esegui ogni cella in ordine con **Maiusc + Invio**.

## Parte 1 — Metriche oltre l'accuratezza

L'accuratezza ("quante ho indovinato di 100?") è semplice, ma nasconde i dettagli:

- Se il modello ha imparato un bias e predice "7" per tutto, avrà bassa accuratezza globale ma potrebbe essere molto bravo con i "7"
- Se un'unica classe è più rara, il modello potrebbe ignorarla per massimizzare l'accuratezza generale

Altre metriche sono più rivelatori:

- **Precision**: delle previsioni "7", quante erano davvero "7"? (falsi positivi minimizzati)
- **Recall**: dei veri "7", quanti li ho indovinati? (falsi negativi minimizzati)
- **F1-score**: media armonica di precision e recall, una metrica bilanciata
- **Matrice di confusione**: una tabella che mostra, per ogni classe vera, come il modello ha predetto

Per un problema di classificazione multiclasse come MNIST, calcola queste metriche per classe e osserva se il modello ha problemi sistematici con una particolare classe.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Versión de PyTorch:", torch.__version__)
print("Dispositivo:", device)

## Parte 2 — Matrice di confusione

Una matrice di confusione è una tabella 10×10 (per MNIST, 10 cifre) dove riga *i* colonna *j* mostra quante immagini di classe *i* il modello ha predetto come classe *j*.

In [ ]:
transform = transforms.ToTensor()

train_dataset_completo = torchvision.datasets.MNIST(
    root="./datos", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(
    root="./datos", train=False, download=True, transform=transform)

N_TRAIN = 2000
N_VAL = 2000

indices_train = list(range(N_TRAIN))
indices_val = list(range(N_TRAIN, N_TRAIN + N_VAL))

train_dataset = Subset(train_dataset_completo, indices_train)
val_dataset = Subset(train_dataset_completo, indices_val)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Ejemplos de entrenamiento: {len(train_dataset)}")
print(f"Ejemplos de validación:   {len(val_dataset)}")
print(f"Ejemplos de test:         {len(test_dataset)}")

## Parte 3 — Visualizzare gli errori

Estraiamo le immagini che il modello ha sbagliato e visualizziamole insieme alla predizione e all'etichetta vera. Spesso, osservando gli errori, capirai perché il modello sbaglia: l'immagine potrebbe essere sfuocata, ambigua, o il modello potrebbe confondere due cifre simili (come 4 e 9).

In [ ]:
class RedDensa(nn.Module):
    def __init__(self):
        super().__init__()
        self.aplanar = nn.Flatten()
        self.capa1 = nn.Linear(28 * 28, 512)
        self.capa2 = nn.Linear(512, 512)
        self.capa_salida = nn.Linear(512, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.aplanar(x)
        x = self.relu(self.capa1(x))
        x = self.relu(self.capa2(x))
        return self.capa_salida(x)


net = RedDensa().to(device)
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(net.parameters(), lr=1e-3)

print(net)

## Parte 4 — Precision, Recall e F1-score per classe

Calcola precision, recall e F1-score per ogni classe. Ciò ti aiuta a identificare quali classi il modello trova difficili.

In [ ]:
def evaluar(modelo, data_loader):
    # Calcula la perdida media y la precision de 'modelo' sobre
    # 'data_loader', sin actualizar sus pesos.
    modelo.eval()
    perdida_total = 0.0
    aciertos = 0
    total = 0
    with torch.no_grad():
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            perdida = funcion_perdida(salidas, etiquetas)

            perdida_total += perdida.item() * etiquetas.size(0)
            predicciones = torch.argmax(salidas, dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)

    return perdida_total / total, aciertos / total


NUM_EPOCHS = 15
historia = {"perdida_train": [], "perdida_val": [],
            "precision_train": [], "precision_val": []}

for epoca in range(NUM_EPOCHS):
    net.train()
    for imagenes, etiquetas in train_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = net(imagenes)
        perdida = funcion_perdida(salidas, etiquetas)
        perdida.backward()
        optimizador.step()

    perdida_train, precision_train = evaluar(net, train_loader)
    perdida_val, precision_val = evaluar(net, val_loader)

    historia["perdida_train"].append(perdida_train)
    historia["perdida_val"].append(perdida_val)
    historia["precision_train"].append(precision_train)
    historia["precision_val"].append(precision_val)

    print(f"Época {epoca + 1:2d}/{NUM_EPOCHS} — "
          f"pérdida train: {perdida_train:.3f}, pérdida val: {perdida_val:.3f} — "
          f"precisión train: {precision_train:.2%}, precisión val: {precision_val:.2%}")

In [ ]:
epocas = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(epocas, historia["perdida_train"], label="Entrenamiento (in-sample)")
plt.plot(epocas, historia["perdida_val"], "--", label="Validación (out-of-sample)")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Pérdida por época")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epocas, historia["precision_train"], label="Entrenamiento (in-sample)")
plt.plot(epocas, historia["precision_val"], "--", label="Validación (out-of-sample)")
plt.xlabel("Época")
plt.ylabel("Precisión")
plt.title("Precisión por época")
plt.legend()

plt.tight_layout()
plt.show()

## Parte 5 — Visualizzazione degli attributi della rete (Layer Activation Visualization)

Per capire quali feature il modello sta usando, visualizza gli output dei filtri convoluzionali. Ogni filtro in una CNN impara un pattern (bordi, texture, ecc.): visualizzarli mostra cosa la rete "vede".

In [ ]:
NUM_CLASES = 10
matriz_confusion = torch.zeros(NUM_CLASES, NUM_CLASES, dtype=torch.int64)
aciertos = 0
total = 0

net.eval()
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        salidas = net(imagenes)
        predicciones = torch.argmax(salidas, dim=1)

        aciertos += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

        for etiqueta_real, etiqueta_predicha in zip(etiquetas.cpu(), predicciones.cpu()):
            matriz_confusion[etiqueta_real, etiqueta_predicha] += 1

precision_test = aciertos / total
print(f"Precisión sobre el conjunto de test: {precision_test:.2%}")

plt.figure(figsize=(6, 5))
plt.imshow(matriz_confusion, cmap="Blues")
plt.colorbar(label="Número de ejemplos")
plt.xlabel("Dígito predicho")
plt.ylabel("Dígito real")
plt.title("Matriz de confusión — conjunto de test")
plt.xticks(range(NUM_CLASES))
plt.yticks(range(NUM_CLASES))
plt.show()

print(matriz_confusion)

## Riepilogo della lezione

- L'accuratezza è semplice ma nasconde i dettagli: una classe potrebbe essere trascurata anche se l'accuratezza globale è alta
- Precision, recall e F1-score ti aiutano a capire le prestazioni per classe
- La matrice di confusione mostra quali classi si confondono tra loro
- Visualizzare gli errori è spesso illuminante: vedrai quali immagini il modello trova difficili e perché
- Visualizzare i filtri convoluzionali aiuta a capire cosa la rete ha imparato
- Queste analisi guidano il design futuro: se una classe è difficile, potrebbe esserci bias nei dati

Nel notebook di pratica estrai gli errori su diverse classi e visualizzerai come il modello confonde specifiche coppie di cifre.

In [ ]:
imagenes_test, etiquetas_test = next(iter(
    DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)))
imagenes_test, etiquetas_test = imagenes_test.to(device), etiquetas_test.to(device)

net.eval()
with torch.no_grad():
    salidas = net(imagenes_test)
    predicciones = torch.argmax(salidas, dim=1)

indices_erroneos = (predicciones != etiquetas_test).nonzero(as_tuple=True)[0]
print(f"Número de predicciones incorrectas sobre el test: {len(indices_erroneos)} de {len(test_dataset)}")

plt.figure(figsize=(10, 2.5))
for i, indice in enumerate(indices_erroneos[:6]):
    indice = indice.item()
    plt.subplot(1, 6, i + 1)
    plt.imshow(imagenes_test[indice].cpu().squeeze(), cmap="gray")
    plt.title(f"Real: {etiquetas_test[indice].item()}\nPred: {predicciones[indice].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Salvare e caricare il modello

Abbiamo già analizzato il modello a fondo. Ora salveremo i suoi pesi con
`torch.save` per non perdere il lavoro, e verificheremo che, ricaricandoli in
un'architettura nuova con `torch.load` e `load_state_dict`, il modello
recuperato predica esattamente come l'originale.

In [ ]:
ruta_modelo = "modelo_clase08.pt"

# Guardamos únicamente los pesos aprendidos (el state_dict), no el objeto
# 'net' completo: es la forma recomendada por la documentación de PyTorch
torch.save(net.state_dict(), ruta_modelo)
print(f"Pesos guardados en '{ruta_modelo}'")

# Reconstruimos la MISMA arquitectura, sin entrenar
modelo_restaurado = RedDensa().to(device)

# Antes de cargar los pesos, sus predicciones deberían ser prácticamente
# aleatorias (alrededor de un 10% de precisión, como elegir al azar entre
# diez dígitos)
_, precision_sin_pesos = evaluar(modelo_restaurado, test_loader)
print(f"Precisión antes de cargar los pesos: {precision_sin_pesos:.2%}")

# Cargamos los pesos guardados dentro de esa arquitectura
modelo_restaurado.load_state_dict(torch.load(ruta_modelo, map_location=device))
modelo_restaurado.eval()

_, precision_con_pesos = evaluar(modelo_restaurado, test_loader)
print(f"Precisión después de cargar los pesos: {precision_con_pesos:.2%}")

# Comprobación más estricta: ¿predicen exactamente lo mismo, ejemplo a
# ejemplo, sobre el mismo lote de imágenes?
with torch.no_grad():
    salida_original = net(imagenes_test)
    salida_restaurada = modelo_restaurado(imagenes_test)

prediccion_original = torch.argmax(salida_original, dim=1)
prediccion_restaurada = torch.argmax(salida_restaurada, dim=1)

print("¿Predicciones idénticas a las del modelo original?",
      torch.equal(prediccion_original, prediccion_restaurada))

## Riepilogo

- Confrontare la perdita e la precisione dentro il campione (addestramento) e
  fuori dal campione (validazione o test) in ogni epoca è il modo più diretto
  per rilevare sovradattamento (le curve si separano) o sotto-adattamento
  (entrambe rimangono alte)
- La precisione di test conta più di quella di addestramento: è l'unica
  cifra che si avvicina a ciò che il modello farà con dati reali
- La matrice di confusione rivela in quali categorie specifiche si sbaglia il
  modello, qualcosa che un'unica percentuale di precisione non mostra
- Osservare esempi concreti di predizioni sbagliate aiuta a capire se
  gli errori del modello sono ragionevoli o inaspettati
- `torch.save(modelo.state_dict(), ruta)` e
  `modelo.load_state_dict(torch.load(ruta))` salvano e recuperano i pesi
  di un modello; devi prima ricostruire la stessa architettura

Continua a esercitarti nel notebook
`clase08_practica_analisis_del_modelo.ipynb` e consulta il capitolo 08 del
manuale per ripassare la teoria più in dettaglio.